# Task 3 — Full-data 학습 (RF-DETR Large, 최종 WBF 앙상블용 체크포인트)

**목적**: task2-masked와 **완전히 동일한 병합 데이터**(원본 train 232장 + 합성 pool2 + masked pool, 74종)를
**split 없이 전부 학습**에 사용해, 최종 제출 WBF 앙상블에 쓸 RF-DETR **Large** 체크포인트를 만든다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | task2-masked와 동일 (corrections 구버전 스냅샷 + `syn_00505/00102` 제외 + masked `msk_` 리네임) — **전량 train** |
| 분할 | **없음** (full data). rfdetr API 형식 요건용 **더미 valid**(74종 커버 최소 샘플, train에도 포함된 중복)만 생성 |
| 모델 | RF-DETR **large**, batch 2 × grad_accum 8 (유효 16 — Large는 16GB에서 batch 4 OOM 가능) |
| 학습량 | **15 epoch 단일 세션** — 20ep 시험 학습에서 dummy-valid 기준 best가 12 epoch에 찍히고 이후 개선 없음(epoch당 ~24분 실측)을 근거로 축소 |
| 체크포인트 | **마지막 epoch** 체크포인트 사용 (더미 valid 기준 best는 무의미 — 고정 epoch 철학) |
| 산출물 | `{tag}_ep{누적}_last.pth` 체크포인트 + 단일모델 test 추론 CSV (`submission_task3_large_task3_full74_masked.csv`) |

**세션 시간 전략**
- 실측: Large × full(≈3,400장)은 **epoch당 약 24분** → 15ep ≈ 학습 6h + 준비/추론 포함 총 7~8h로
  **12h 단일 세션 안에 완료** (청크 분할 불필요, cosine LR 스케줄도 한 번에 온전히 적용됨)
- 학습 셀의 청크(이어하기) 변수는 남겨둠 — 나중에 epoch을 더 늘리고 싶을 때
  `EPOCHS_DONE_BEFORE`/`INIT_CHECKPOINT`로 이어서 학습 가능

**실행 전제**
- Kaggle Notebook: **GPU on / Internet on**
- Input 연결: competition 데이터(`train_images`, `train_annotations`, `test_images`) + `task2_synthesized` + `dataset-74-masked`
  (fold 분할이 없으므로 `fold_split_masked.json`은 필요 없음)
- 원칙: **저장소에 등록된 파일/함수는 수정하지 않는다.** 그대로 쓰기 어려운 부분만 노트북 내 로컬 코드로 우회한다.


In [ ]:
# [1. 환경 준비] 필요 패키지 설치
#  - rfdetr[train]  : RF-DETR 학습/추론 (metrics.csv 로깅 포함)
#  - torchmetrics   : 저장소 visualize.py가 사용
#  (task2와 달리 ensemble-boxes는 설치하지 않음 - 단일 모델이라 WBF 융합이 없음)
#  ⚠ pip 설치는 세션이 재시작되면 사라지므로 매 세션 이 셀부터 실행해야 합니다.
#  rfdetr 버전 고정: 체크포인트 파일명 체계(checkpoint_best_total.pth 등)를 소스에서 확인한 버전.
#  버전을 풀면 릴리스에 따라 파일명이 바뀌어 백업 셀이 깨질 수 있음 (실제 사고 사례 있음).
!pip install -q "rfdetr[train]==1.8.3" torchmetrics

# 설치 검증: -q 옵션이 에러 로그를 가리는 경우 대비
import importlib
for m in ('rfdetr', 'torchmetrics'):
    mod = importlib.import_module(m)
    print('설치 확인 OK:', m, getattr(mod, '__version__', ''))


In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다.
import os, sys, re, glob, json, shutil, time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/kaggle/working/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 목록 ─────────────────────────────
from dataset import (
    load_raw_annotations,     # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,        # corrections.json 기반 bbox 수정
    build_coco,               # 파일 목록 -> COCO dict (fold 없이 train/valid 디렉토리를 직접 구성하는 데 사용)
    cache_images,             # 이미지 로컬 캐시
    save_label_map,           # label_map.json 저장 (앙상블 노트북에서 라벨 매핑 참조용)
)
from model import get_rfdetr_model                    # RF-DETR variant 생성 ('large' 지원, checkpoint_path 로드)
from utils import read_metrics_csv, plot_history      # 학습곡선 출력/저장
from visualize import collect_predictions_ensemble    # 모델 리스트 추론 - 단일 모델도 [model]로 재사용
# make_folds / write_fold_dirs / run_kfold은 사용하지 않습니다 (task3는 split 없이 full data 학습)


In [ ]:
# [corrections 하드코딩] 저장소의 corrections.json을 파일에서 읽지 않고 노트북에 직접 정의합니다.
#  apply_corrections()는 "파일 경로"를 받는 시그니처라(저장소 함수 무수정 원칙),
#  dict를 /kaggle/working에 json으로 1회 저장한 뒤 그 경로를 넘기는 방식으로 우회합니다.
#  적용 순서(coord_fix -> remove -> modify -> add -> fix_category)는 함수 내부에서 보장됩니다.
corrections = {
    # 1) 좌표 오염 수정: 동일 bbox를 가진 항목을 corrected로 교체
    "coord_fix": {
        "K-003351-016262-018357_0_2_0_2_75_000_200.png": [
            {"original": [6567, 625, 311, 315], "corrected": [567, 625, 311, 315]}
        ]
    },
    # 2) 중복/오류 박스 제거: category_id + bbox가 일치하는 첫 항목 1개 제거
    "remove_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [88, 255, 366, 209]}
        ]
    },
    # 3) 좌표 수정: category_id(+match_bbox) 첫 매치의 bbox를 교체하거나 directive 적용
    "modify_boxes": {
        "K-003351-020014-020238_0_2_0_2_90_000_200.png": [
            {"category_id": 3351, "match_bbox": None, "new_bbox": [390, 260, 170, 165]}
        ],
        "K-003351-019232-029667_0_2_1_2_70_000_200.png": [
            {"category_id": 19232, "match_bbox": None, "directive": "EXTEND_DOWN_95"}
        ]
    },
    # 4) 누락 박스 추가 (원본에 없던 박스라 annotation_id는 None으로 채워짐)
    "add_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [90, 870, 245, 240]}
        ],
        "K-003351-013900-021325_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [400, 830, 180, 180]}
        ],
        "K-003351-013900-036637_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [440, 880, 175, 175]}
        ],
        "K-003351-020014-022074_0_2_0_2_90_000_200.png": [
            {"category_id": 20014, "bbox": [65, 720, 325, 315]}
        ],
        "K-003351-020238-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 20238, "bbox": [580, 290, 215, 215]}
        ],
        "K-003351-021325-032310_0_2_0_2_90_000_200.png": [
            {"category_id": 32310, "bbox": [595, 830, 345, 245]}
        ],
        "K-003351-029667-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [375, 870, 165, 165]}
        ],
        "K-003351-032310-038162_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [390, 855, 185, 185]}
        ],
        "K-003351-033880-038162_0_2_0_2_75_000_200.png": [
            {"category_id": 33880, "bbox": [70, 600, 310, 425]}
        ],
        "K-003351-035206-041768_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [460, 875, 180, 180]}
        ],
        "K-003544-004543-012247-016548_0_2_0_2_90_000_200.png": [
            {"category_id": 4543, "bbox": [640, 195, 205, 190]}
        ]
    },
    # 5) category_id 오기재 수정: 원본 annotation_id -> 올바른 category_id
    #    (키는 문자열이어야 함 - apply_corrections 내부에서 int(k)로 변환)
    "fix_category": {
        "791": 31863,
        "3444": 3351,
        "3441": 3351,
        "1420": 35206,
        "1412": 27733
        }
    }

CORRECTIONS_PATH = '/kaggle/working/corrections.json'
with open(CORRECTIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(corrections, f, ensure_ascii=False, indent=2)
    
def count_items(v):
    if isinstance(v, dict):
        try:
            return sum(len(i) if hasattr(i, '__len__') else 1 for i in v.values())
        except:
            return len(v)
    elif hasattr(v, '__len__'):
        return len(v)
    else:
        return 1

print('corrections 하드코딩 저장:', CORRECTIONS_PATH,
      '| 항목:', {k: count_items(corrections[k]) for k in corrections})

In [ ]:
# [3. 입력 경로 자동 탐색]
#  /kaggle/input 아래에서 폴더 "이름"으로 찾기 때문에 competition/데이터셋 슬러그를 몰라도 동작합니다.
#  탐색 실패 시 None이 출력되므로, 그 경우 아래 상수에 실제 경로를 직접 채워 넣으세요.
def find_input_dir(name):
    """/kaggle/input 아래에서 이름이 name인 디렉토리를 찾아 반환합니다 (여러 개면 첫 번째)."""
    hits = sorted(p for p in glob.glob(os.path.join('/kaggle/input', '**', name), recursive=True)
                  if os.path.isdir(p))
    if len(hits) > 1:
        print(f"'{name}' 후보 {len(hits)}개 -> 첫 번째 사용:\n  " + "\n  ".join(hits))
    return hits[0] if hits else None

TRAIN_IMG_DIR = find_input_dir('train_images')       # 원본 train 이미지 (png)
TRAIN_ANN_DIR = find_input_dir('train_annotations')  # 원본 annotation (박스당 json 1개)
TEST_IMG_DIR  = find_input_dir('test_images')        # 제출 대상 test 이미지

POOL_DIR      = find_input_dir('task2_synthesized')   # 합성 pool 루트 (images/, annotations/)
POOL_IMG_DIR  = os.path.join(POOL_DIR, 'images') if POOL_DIR else None
POOL_ANN_PATH = ((glob.glob(os.path.join(POOL_DIR, 'annotations', '*.json')) or [None])[0]
                 if POOL_DIR else None)

MASKED_DIR      = find_input_dir('dataset-74-masked')  # masked pool 루트 (images/, annotations/)
MASKED_IMG_DIR  = os.path.join(MASKED_DIR, 'images') if MASKED_DIR else None
MASKED_ANN_PATH = ((glob.glob(os.path.join(MASKED_DIR, 'annotations', '*.json')) or [None])[0]
                   if MASKED_DIR else None)

paths = {'TRAIN_IMG_DIR': TRAIN_IMG_DIR, 'TRAIN_ANN_DIR': TRAIN_ANN_DIR, 'TEST_IMG_DIR': TEST_IMG_DIR,
         'POOL_IMG_DIR': POOL_IMG_DIR, 'POOL_ANN_PATH': POOL_ANN_PATH,
         'MASKED_IMG_DIR': MASKED_IMG_DIR, 'MASKED_ANN_PATH': MASKED_ANN_PATH}
for k, v in paths.items():
    print(f'{k:15s}:', v)
assert all(paths.values()), "자동 탐색에 실패한 경로가 있습니다. 위 상수에 직접 경로를 지정하세요."

In [ ]:
# [4. 실험 설정]
#  - RF-DETR large: 16GB GPU에서 batch 4는 OOM 가능성이 높아 batch 2 x grad_accum 8 = 유효 배치 16 유지
#  - epochs는 config에 두지 않습니다: 세션 청크 단위로 학습하므로 11번(학습) 셀에서 세션마다 지정
#  - early_stopping=False: 더미 valid 기준 지표로 학습을 멈추면 안 되므로 반드시 비활성
TASK_ID = 3

config = {
    'data': {
        'dataset_dir': '/kaggle/tmp/dataset_full',   # train/ + valid(더미)/ 구조 (fold 없음)
        'cache_dir':   '/kaggle/tmp/img_cache',
    },
    'model': {
        'variant': 'large',
        'tag': 'large_task3_full74_masked',
    },
    'train': {
        'batch_size': 2,
        'grad_accum_steps': 8,
        'lr': 1e-4,
        'lr_encoder': 1.5e-4,
        'weight_decay': 1e-4,
        'lr_scheduler': 'cosine',
        'warmup_epochs': 0.0,
        'lr_min_factor': 0.0,
        'early_stopping': False,
        'early_stopping_patience': 10,     # early_stopping=False라 미사용이지만 train 시그니처상 필요
        'early_stopping_min_delta': 0.001,
        'tensorboard': False,
    },
    'output': {
        'local_output_dir': '/kaggle/tmp/outputs',
        'backup_dir': '/kaggle/working/outputs',   # 체크포인트/학습곡선 (커밋 시 보존)
    },
}

COLLECT_SCORE_THR = 0.05   # test 추론 수집 threshold (시각화에서 낮은 score도 점검할 수 있게 낮게)
SUBMIT_SCORE_THR  = 0.3    # 제출 CSV 포함 최소 confidence

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['output']['local_output_dir'], config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

SUBMISSION_PATH = f"/kaggle/working/submission_task{TASK_ID}_{config['model']['tag']}.csv"
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 가속기 설정 확인')
print('제출 파일 경로:', SUBMISSION_PATH)


In [ ]:
# [5. 원본 train 로드 + annotation 수정]
#  원본은 "박스 1개당 JSON 1개"로 흩어져 있어 load_raw_annotations()가 파일명 기준으로 병합합니다.
#  이어서 corrections.json(팀에서 검수한 좌표 오류/중복/누락/클래스 오기재 내역)을 적용합니다.
#  (apply_corrections는 coord_fix -> remove -> modify -> add -> fix_category 순서를 내부에서 보장)
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(TRAIN_ANN_DIR)
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

# 원본 train의 클래스 목록(56종)을 보존해 둡니다.
#  - Task1: pool이 이 범위를 벗어나지 않는지 검증에 사용
#  - Task2: "train 56종 -> 라벨 1~56" 매핑 검증에 사용
train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))

In [ ]:
# [6. 합성 pool 병합] pool2 (test 전용 18종 + train 희소 클래스 균형 배치)
#  pool의 _annotations.coco.json은 "라벨 네임스페이스"(id 1~N, name=원본 category_id)로 작성되어 있습니다.
#  train 쪽 파이프라인 함수(build_coco 등)는 "원본 category_id" 기준으로 동작하므로,
#  categories의 name 필드를 이용해 원본 id 공간으로 되돌린 뒤 병합합니다. (로컬 함수로 우회)
#  annotation id도 함께 수집합니다 (시각화에서 ann_id 표시용. pool JSON 자체의 id라서
#  train의 원본 annotation id와 번호가 겹칠 수 있으나, 표시 용도로만 쓰므로 무방).
def load_pool_annotations(pool_ann_path):
    """합성 pool COCO json -> (boxes, cats(원본 id 공간), ids, img_meta, 원본 coco dict)"""
    with open(pool_ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    # 라벨 -> 원본 category_id (name 필드가 원본 id 문자열, id 0은 RF-DETR용 더미 'pill')
    label2cat_pool = {c['id']: int(c['name']) for c in coco['categories'] if c['id'] != 0}
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    p_boxes, p_cats, p_ids, p_meta = defaultdict(list), defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        p_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        p_boxes[fn].append([float(v) for v in a['bbox']])
        p_cats[fn].append(label2cat_pool[a['category_id']])
        p_ids[fn].append(a.get('id'))
    return p_boxes, p_cats, p_ids, p_meta, coco

pool_boxes, pool_cats, pool_ids, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

# 안전장치 1: 파일명 충돌 확인 (syn_*.png vs K-*.png 라 충돌 없어야 정상)
overlap = set(pool_meta) & set(boxes_by_image)
assert not overlap, f'train/pool 파일명 충돌: {sorted(overlap)[:5]}'

# 안전장치 2: 박스가 하나도 없는 pool 이미지는 학습에 기여하지 못하므로 제외
empty_pool = [fn for fn in pool_meta if not pool_boxes.get(fn)]
if empty_pool:
    print(f'박스 0개인 pool 이미지 {len(empty_pool)}장 제외:', empty_pool[:5])

for fn in pool_meta:
    if pool_boxes.get(fn):
        boxes_by_image[fn] = pool_boxes[fn]
        cats_by_image[fn] = pool_cats[fn]
        ids_by_image[fn] = pool_ids[fn]   # 시각화(ann_id 표시)에서 사용
        img_meta[fn] = pool_meta[fn]

print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

In [ ]:
# [6-1. masked pool 병합] dataset-74-masked (실사 기반 masked pool)
#  masked pool의 _annotations.coco.json은 합성 pool과 달리 categories의 id가 "원본 category_id 그대로"라
#  name 매핑 없이 바로 사용합니다.
#  파일명이 train set과 동일 체계(K코드+촬영조건)라 원본 train 232장과 겹칠 수 있으므로,
#  전체 파일에 'msk_' 접두어를 붙여 리네임한 사본을 스테이징 폴더에 만들어 사용합니다.
#   - cache_images()가 파일명 기준으로 한 폴더에 복사하므로 리네임 없이는 이미지가 덮어써짐
#   - corrections(원본 파일명이 키)가 masked 사본에 잘못 적용되는 일도 방지
#   - task3는 split이 없어 그룹화가 불필요하지만, task2-masked와 데이터 처리(리네임 포함)를 동일하게 유지합니다
MASKED_STAGE_DIR = '/kaggle/tmp/masked_renamed'
os.makedirs(MASKED_STAGE_DIR, exist_ok=True)

def load_masked_annotations(ann_path):
    """masked pool COCO json -> (boxes, cats, ids, meta, coco). category_id를 원본 id로 그대로 사용합니다."""
    with open(ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    m_boxes, m_cats, m_ids, m_meta = defaultdict(list), defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        m_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        m_boxes[fn].append([float(v) for v in a['bbox']])
        m_cats[fn].append(int(a['category_id']))
        m_ids[fn].append(a.get('id'))
    return m_boxes, m_cats, m_ids, m_meta, coco

masked_boxes, masked_cats, masked_ids, masked_meta, masked_coco = load_masked_annotations(MASKED_ANN_PATH)
print(f"masked pool: 이미지 {len(masked_meta)}장 / 박스 {sum(len(v) for v in masked_boxes.values())}개"
      f" / 클래스 {len({c for cs in masked_cats.values() for c in cs})}종")

# 안전장치 1: masked pool의 category_id가 74종 매핑(원본 id 공간 = pool2 JSON의 name)에 전부 포함되는지 확인
#  -> 라벨 공간(1~74) json을 잘못 넣은 경우 등 포맷 착오를 학습 전에 잡습니다.
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
unknown = sorted({c for cs in masked_cats.values() for c in cs} - cats_74)
assert not unknown, f'74종 매핑에 없는 masked pool category_id: {unknown} (원본 id/라벨 공간 착오 확인)'

# 안전장치 2: annotation에는 있는데 실제 이미지 파일이 없는 항목 확인
masked_img_src = {os.path.basename(p): p
                  for p in glob.glob(os.path.join(MASKED_IMG_DIR, '**', '*.png'), recursive=True)}
missing_img = sorted(set(masked_meta) - set(masked_img_src))
assert not missing_img, f'annotation에는 있는데 이미지가 없는 파일: {missing_img[:5]}'

# 안전장치 3: 박스 0개 이미지는 제외 (task2-masked와 동일 기준 - 학습 기여 없음)
empty_masked = [fn for fn in masked_meta if not masked_boxes.get(fn)]
if empty_masked:
    print(f'박스 0개인 masked 이미지 {len(empty_masked)}장 제외:', empty_masked[:5])

# 리네임('msk_' + 원본 파일명) + 스테이징 복사 + annotation 병합
n_merged = 0
for fn in masked_meta:
    if not masked_boxes.get(fn):
        continue
    new_fn = 'msk_' + fn
    assert new_fn not in boxes_by_image, f'리네임 후에도 파일명 충돌: {new_fn}'
    dst = os.path.join(MASKED_STAGE_DIR, new_fn)
    if not os.path.exists(dst):
        shutil.copy(masked_img_src[fn], dst)
    boxes_by_image[new_fn] = masked_boxes[fn]
    cats_by_image[new_fn] = masked_cats[fn]
    ids_by_image[new_fn] = masked_ids[fn]   # masked JSON의 id (시각화 표시 용도)
    img_meta[new_fn] = masked_meta[fn]
    n_merged += 1

print(f'masked pool {n_merged}장 리네임 병합 완료 (스테이징: {MASKED_STAGE_DIR})')
print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [6-3. 데이터 제외] task2-masked와 동일한 제외 목록을 그대로 적용합니다 (데이터 동일성 유지).
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]

for fn in EXCLUDE_FILES:
    if fn in boxes_by_image:
        for d in (boxes_by_image, cats_by_image, ids_by_image, img_meta):
            d.pop(fn, None)
        print('이미지 제외:', fn)
    else:
        print('⚠ 목록에 없는 파일명(오타 확인):', fn)

print(f"제외 처리 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [8. 카테고리 라벨 매핑 - Task 2 전용: 74종]
#  요구 체계: train 56종 -> 라벨 1~56, test 전용 18종 -> 라벨 57~74.
#  pool2 JSON의 categories가 이미 이 체계(1~74, name=원본 category_id)로 작성되어 있으므로
#  이를 신뢰 소스로 사용합니다.
#  ⚠ 저장소 build_category_mapping()을 74종에 그대로 쓰면 "등장 클래스 전체 오름차순"이라
#    test 18종이 1~56 사이에 끼어들어 요구 체계가 깨지므로, 여기서는 로컬 매핑으로 우회합니다.
cat2label = {int(c['name']): c['id'] for c in pool_coco['categories'] if c['id'] != 0}
label2cat = {v: k for k, v in cat2label.items()}
all_cats = [label2cat[l] for l in sorted(label2cat)]   # build_coco()에 넘길 카테고리 목록(라벨 오름차순)

# 검증 1: train 56종이 정확히 1~56에 (오름차순으로) 매핑되는가 -> Task 0/1과 네임스페이스 호환 보장
for i, c in enumerate(sorted(train_cats), start=1):
    assert cat2label.get(c) == i, f'train 클래스 {c}가 라벨 {cat2label.get(c)}에 매핑됨 (기대: {i})'

# 검증 2: 병합 데이터에 등장하는 모든 클래스가 매핑에 존재하는가
merged_cats = {c for cs in cats_by_image.values() for c in cs}
missing = sorted(merged_cats - set(cat2label))
assert not missing, f'매핑에 없는 클래스 존재: {missing}'

test_only_labels = sorted(set(cat2label.values()) - set(range(1, len(train_cats) + 1)))
print(f'전체 {len(cat2label)}종 매핑 | train {len(train_cats)}종 -> 1~{len(train_cats)} 확인'
      f' | test 전용 라벨: {test_only_labels}')

In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [9. full 데이터셋 디렉토리 생성] fold 분할 없음 - 전체 데이터를 train에 배치합니다.
#  rfdetr의 model.train()은 dataset_dir 아래 train/, valid/ 폴더 구조를 "형식상" 요구합니다.
#   - train: 병합 데이터 전체 (한 장도 빼지 않음)
#   - valid: 더미 - 74종을 커버하는 최소 샘플 (train에도 그대로 포함된 중복본)
#     -> valid 지표는 train 부분집합 기준이라 성능 지표가 아니며, epoch당 평가 시간을
#        최소화하기 위해 크기를 최소로 유지합니다. 체크포인트는 best가 아닌 "마지막 epoch" 사용.
file_names = sorted(boxes_by_image)

# 더미 valid: 각 라벨이 최소 1번 등장하도록 그리디 선택
need = set(cat2label)   # 아직 커버되지 않은 원본 category_id
dummy_valid = []
for fn in file_names:
    hit = need & set(cats_by_image[fn])
    if hit:
        dummy_valid.append(fn)
        need -= hit
    if not need:
        break
assert not need, f'더미 valid가 커버하지 못한 클래스: {sorted(need)}'
print(f'train: {len(file_names)}장 (전체) / 더미 valid: {len(dummy_valid)}장 (74종 커버)')

cache_images(TRAIN_IMG_DIR, config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])

for split, files in (('train', file_names), ('valid', dummy_valid)):
    d = os.path.join(config['data']['dataset_dir'], split)
    os.makedirs(d, exist_ok=True)
    coco = build_coco(files, boxes_by_image, cats_by_image, img_meta, cat2label, all_cats)
    with open(os.path.join(d, '_annotations.coco.json'), 'w') as f:
        json.dump(coco, f)
    for fn in files:
        shutil.copy(os.path.join(config['data']['cache_dir'], fn), os.path.join(d, fn))
    print(f'{split}: 이미지 {len(files)}장 / 박스 {len(coco["annotations"])}개')

# 라벨 매핑을 backup_dir에 저장 - 최종 앙상블 노트북에서 이 체크포인트의 라벨 체계를 참조할 때 사용
save_label_map(cat2label, label2cat, config['output']['backup_dir'])

# 캐시는 복사가 끝나면 불필요 - 삭제해 디스크 확보
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('데이터셋 준비 완료:', config['data']['dataset_dir'])


## 10~11. full data 학습 (15 epoch 단일 세션)

20ep 시험 학습(epoch당 ~24분)에서 dummy-valid 기준 **best가 12 epoch**에 찍히고 이후 개선이 없었다.
이를 근거로 **15 epoch 단일 세션**으로 학습한다 — 학습 ~6h + 준비/추론 포함 총 7~8h로 12h 안에 완료.

- **산출물**: `large_task3_full74_masked_ep15_last.pth` — 실제 내용물은 `checkpoint_best_total.pth`
  (더미 valid 기준 best, regular vs EMA 중 나은 쪽)의 사본. best가 12ep 부근이면 그 시점 가중치가 담긴다.
  (파일명의 ep15는 "이 run의 학습량"을 뜻함 — 앙상블 노트북의 `ep*_last.pth` 검색 패턴과 호환)
- **나중에 epoch을 늘리고 싶으면**: 이 커밋 Output을 Input으로 추가하고
  `EPOCHS_DONE_BEFORE=15, EPOCHS_THIS_SESSION=<추가분>, INIT_CHECKPOINT=<ep15 체크포인트 경로>`로 이어하기
  (`pretrain_weights` 방식이라 cosine LR 스케줄은 청크마다 재시작됨).


In [ ]:
# [10~11. full data 학습] 15 epoch 단일 세션.
#  근거: 20ep 시험 학습에서 best가 12 epoch에 찍히고 이후 개선 없음 (epoch당 약 24분 실측)
#  -> 15ep ≈ 학습 6h, 단일 세션으로 완료. 청크(이어하기) 변수는 추후 연장 학습용으로 남겨둡니다.
TARGET_EPOCHS = 15           # 총 목표 epoch (시험 학습 best=12ep 관찰 기반 + 여유 3ep)
EPOCHS_DONE_BEFORE = 0       # 이어하기 시: 이전 세션까지 누적 완료한 epoch 수
EPOCHS_THIS_SESSION = 15     # 이번 세션에 학습할 epoch 수
INIT_CHECKPOINT = None       # 이어하기 시 이전 체크포인트 경로 (None이면 COCO pretrained에서 시작)
#  연장 학습 예: EPOCHS_DONE_BEFORE=15, EPOCHS_THIS_SESSION=5,
#      INIT_CHECKPOINT = '/kaggle/input/<이번-커밋-슬러그>/outputs/large_task3_full74_masked_ep15_last.pth'

assert EPOCHS_DONE_BEFORE + EPOCHS_THIS_SESSION <= TARGET_EPOCHS, 'TARGET_EPOCHS 초과 - 값 확인'
assert (EPOCHS_DONE_BEFORE == 0) == (INIT_CHECKPOINT is None), (
    '이어하기면 EPOCHS_DONE_BEFORE>0 + INIT_CHECKPOINT 지정, 첫 세션이면 둘 다 초기값(0/None)이어야 합니다')

exp = config['model']['tag']
out = os.path.join(config['output']['local_output_dir'], exp)
os.makedirs(out, exist_ok=True)

model = get_rfdetr_model(config['model']['variant'], checkpoint_path=INIT_CHECKPOINT)
t0 = time.time()
model.train(
    dataset_dir=config['data']['dataset_dir'],
    output_dir=out,
    epochs=EPOCHS_THIS_SESSION,
    batch_size=config['train']['batch_size'],
    grad_accum_steps=config['train']['grad_accum_steps'],
    lr=config['train']['lr'],
    lr_encoder=config['train']['lr_encoder'],
    weight_decay=config['train']['weight_decay'],
    lr_scheduler=config['train']['lr_scheduler'],
    warmup_epochs=config['train']['warmup_epochs'],
    lr_min_factor=config['train']['lr_min_factor'],
    early_stopping=config['train']['early_stopping'],
    early_stopping_patience=config['train']['early_stopping_patience'],
    early_stopping_min_delta=config['train']['early_stopping_min_delta'],
    tensorboard=config['train']['tensorboard'],
)
elapsed = time.time() - t0
remain = TARGET_EPOCHS - EPOCHS_DONE_BEFORE - EPOCHS_THIS_SESSION
per_ep = elapsed / EPOCHS_THIS_SESSION
print(f"\n이번 세션 {EPOCHS_THIS_SESSION} epoch 소요: {elapsed/3600:.2f}h (epoch당 {per_ep/60:.1f}분)")
if remain:
    print(f"남은 {remain} epoch 예상: {remain*per_ep/3600:.1f}h")

# 체크포인트 백업 - rfdetr 1.8.x(PyTorch Lightning 스택)의 실제 출력 파일명 기준.
#  출력물: last.ckpt(매 epoch 갱신, optimizer 포함 대용량) / checkpoint_epoch=N.ckpt(10ep 간격 아카이브)
#          checkpoint_best_regular.pth / checkpoint_best_ema.pth / checkpoint_best_total.pth
#  이 중 checkpoint_best_total.pth(regular vs EMA 중 나은 쪽, 가중치만 stripped)를 산출물로 사용합니다.
#   - 더미 valid(train 부분집합) 기준 best라 사실상 "후반 수렴 epoch" 선택과 같으며,
#     task2 체크포인트와 동일한 포맷이라 get_rfdetr_model(pretrain_weights) 로드가 검증되어 있습니다.
#  ⚠ 백업 복사가 최우선 - 보조 산출물(학습곡선)은 그 뒤에서 실패해도 무해하게 처리합니다.
cum = EPOCHS_DONE_BEFORE + EPOCHS_THIS_SESSION
print('학습 출력 파일:', sorted(os.listdir(out)))
_cands = ['checkpoint_best_total.pth', 'checkpoint_best_regular.pth', 'checkpoint.pth']
last_src = next((os.path.join(out, c) for c in _cands if os.path.exists(os.path.join(out, c))), None)
if last_src:
    BACKUP_CKPT = os.path.join(config['output']['backup_dir'], f'{exp}_ep{cum:02d}_last.pth')
    shutil.copy(last_src, BACKUP_CKPT)
    print(f'체크포인트 백업: {os.path.basename(last_src)} -> {BACKUP_CKPT}')
else:
    # ⚠ 예상 파일명이 하나도 없는 경우 - 여기서 raise하면 Kaggle 실패 커밋은 output을 보존하지 않아
    #   학습 결과가 통째로 유실됩니다. raise 대신 산출물 전체를 백업해 무조건 뭔가는 남게 합니다.
    print('⚠ 예상 체크포인트 파일명을 찾지 못함 - 학습 산출물 전체를 통째로 백업합니다 (위 파일 목록 확인)')
    copied = []
    for p in sorted(glob.glob(os.path.join(out, '*.pth')) + glob.glob(os.path.join(out, '*.ckpt'))):
        shutil.copy(p, config['output']['backup_dir'])
        copied.append(os.path.basename(p))
        print('  백업:', os.path.basename(p))
    # 이후 추론 셀이 계속 진행되도록 best로 추정되는 파일을 골라둠 (없으면 첫 .pth)
    _best_like = [c for c in copied if 'best' in c and c.endswith('.pth')]
    _pths = [c for c in copied if c.endswith('.pth')]
    BACKUP_CKPT = (os.path.join(config['output']['backup_dir'], (_best_like or _pths)[0])
                   if (_best_like or _pths) else None)
    print('추론에 사용할 추정 체크포인트:', BACKUP_CKPT)

# (선택) 정확한 "마지막 epoch" 상태(last.ckpt)도 보존하려면 주석 해제 (optimizer 포함이라 수 GB - 커밋 시간 증가)
# if os.path.exists(os.path.join(out, 'last.ckpt')):
#     shutil.copy(os.path.join(out, 'last.ckpt'),
#                 os.path.join(config['output']['backup_dir'], f'{exp}_ep{cum:02d}_last.ckpt'))

# 학습곡선 저장 - 실패해도 체크포인트 백업을 막지 않도록 try로 감쌉니다
#  (1.8.x의 metrics.csv는 PTL CSVLogger 포맷이라 저장소 read_metrics_csv와 컬럼이 다를 수 있음)
try:
    if os.path.exists(os.path.join(out, 'metrics.csv')):
        history = read_metrics_csv(out)
        plot_history(history, title=f'{exp} ep{EPOCHS_DONE_BEFORE + 1}~{cum}',
                     save_path=os.path.join(config['output']['backup_dir'], f'{exp}_ep{cum:02d}_history.png'))
    else:
        print('metrics.csv 없음 - 학습곡선 생략')
except Exception as e:
    print('학습곡선 생성 실패 (체크포인트 백업은 완료됨):', e)


In [ ]:
# [12. test 추론 - 단일 모델] 누적 TARGET_EPOCHS에 도달한 최종 체크포인트로 추론합니다.
#  단일 모델이므로 fold 앙상블/WBF 융합이 없습니다 (DETR 계열이라 NMS도 불필요).
FINAL_CKPT = BACKUP_CKPT   # 이번 세션에 만든 체크포인트. 이전 세션 최종본을 쓰려면 경로를 직접 지정
if FINAL_CKPT is None:
    raise SystemExit('사용할 체크포인트가 없어 추론을 건너뜁니다 (백업된 원본 파일로 다음 세션에서 추론 가능)')
print('사용 체크포인트:', FINAL_CKPT)

model = get_rfdetr_model(config['model']['variant'], checkpoint_path=FINAL_CKPT)
test_pred_data = collect_predictions_ensemble([model], TEST_IMG_DIR,
                                              score_threshold=COLLECT_SCORE_THR)
print('test 이미지 수:', len(test_pred_data))

# RF-DETR 내부 예약 라벨(배경 0 등) 제거 - task2와 동일한 정제 과정
valid_labels = set(label2cat)
for d in test_pred_data:
    keep = torch.tensor([int(l) in valid_labels for l in d['pred_labels']], dtype=torch.bool)
    for k in ('pred_boxes', 'pred_labels', 'pred_scores', 'pred_fold'):
        d[k] = d[k][keep]
print('예측 박스 수:', sum(len(d['pred_boxes']) for d in test_pred_data))

# 이후 시각화/CSV 셀은 task2와 동일한 코드를 재사용하므로 변수 이름만 맞춰줍니다.
fused_data = test_pred_data


In [ ]:
# [13. 추론 결과 클래스별 시각화] 단일 모델 예측을 클래스별 crop grid로 표시합니다 (task2와 동일 함수).
#  각 crop 제목에 confidence score + 파일명이 표시됩니다.
#  저장소 crop_predictions_by_class() 대신 task2와 동일한 로컬 함수를 재사용합니다.
def show_pred_class_crops(fused_data, label2cat, score_thr=0.3, max_per_class=8, pad=8):
    """예측을 클래스별 crop grid로 표시합니다 (제목: confidence / 파일명).

    Args:
        fused_data: fuse_predictions_wbf()의 반환값
        label2cat (dict): 모델 라벨 -> 원본 category_id
        score_thr (float): 표시할 예측의 최소 confidence (제출 기준과 동일하게 두고 점검 권장)
        max_per_class (int): 클래스당 표시할 crop 수 (score 내림차순 상위)
        pad (int): crop 여백(px)
    """
    by_label = defaultdict(list)
    for d in fused_data:
        h, w = d['image'].shape[:2]
        keep = d['pred_scores'] >= score_thr
        for box, label, score in zip(d['pred_boxes'][keep], d['pred_labels'][keep],
                                     d['pred_scores'][keep]):
            x1, y1, x2, y2 = box.tolist()
            x1, y1 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            x2, y2 = min(w, int(x2) + pad), min(h, int(y2) + pad)
            if x2 <= x1 or y2 <= y1:
                continue
            by_label[int(label)].append((d['image'][y1:y2, x1:x2], float(score), d['file_name']))

    print(f'score >= {score_thr} 기준, 예측이 존재하는 클래스: {len(by_label)}개')
    for label in sorted(by_label):
        items = sorted(by_label[label], key=lambda t: -t[1])[:max_per_class]
        fig, axes = plt.subplots(1, max_per_class, figsize=(2.2 * max_per_class, 2.6))
        axes = np.atleast_1d(axes)
        for ax in axes:
            ax.axis('off')
        for ax, (crop, score, fn) in zip(axes, items):
            ax.imshow(crop)
            ax.set_title(f'{score:.2f}\n{fn[:16]}', fontsize=6)
        fig.suptitle(f'label={label} / category_id={label2cat[label]}'
                     f'  (score>={score_thr}: {len(by_label[label])}건)', fontsize=9, y=1.05)
        plt.tight_layout()
        plt.show()

show_pred_class_crops(fused_data, label2cat, score_thr=SUBMIT_SCORE_THR, max_per_class=8)

In [ ]:
# [14. 제출 CSV 생성] task2와 동일한 로직/포맷입니다 (단일 모델 예측 기준).
#  요구 포맷: annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score
#   - image_id: 이미지 "파일명의 숫자"
#   - category_id: 원본 category_id (모델 라벨 -> label2cat 역매핑)
#   - annotation_id: 행마다 고유한 임의 값 (여기서는 1부터 증가)
#   - bbox: xyxy(내부 표현) -> xywh(COCO)로 변환
def extract_image_id(file_name):
    """파일명에서 숫자를 추출해 image_id로 사용합니다.
    숫자 블록이 2개 이상이면 어떤 규칙인지 판단할 수 없으므로 일부러 에러를 내어 확인을 요구합니다.
    (그 경우 test 파일명 규칙에 맞게 이 함수만 수정하면 됩니다)
    """
    stem = os.path.splitext(file_name)[0]
    digits = re.findall(r'\d+', stem)
    assert len(digits) == 1, f'파일명 숫자 규칙 확인 필요: {file_name} -> {digits}'
    return int(digits[0])

def make_submission(fused_data, label2cat, score_thr, out_path):
    """예측을 제출 포맷 DataFrame으로 만들어 CSV 저장합니다. (로컬 정의)"""
    rows, ann_id, n_empty = [], 1, 0
    for d in fused_data:
        image_id = extract_image_id(d['file_name'])
        keep = d['pred_scores'] >= score_thr
        if int(keep.sum()) == 0:
            n_empty += 1
        for box, label, score in zip(d['pred_boxes'][keep], d['pred_labels'][keep],
                                     d['pred_scores'][keep]):
            x1, y1, x2, y2 = box.tolist()
            rows.append({
                'annotation_id': ann_id,
                'image_id': image_id,
                'category_id': label2cat[int(label)],
                'bbox_x': round(x1, 2), 'bbox_y': round(y1, 2),
                'bbox_w': round(x2 - x1, 2), 'bbox_h': round(y2 - y1, 2),
                'score': round(float(score), 4),
            })
            ann_id += 1
    df = pd.DataFrame(rows, columns=['annotation_id', 'image_id', 'category_id',
                                     'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score'])
    df.to_csv(out_path, index=False)
    print(f'저장 완료: {out_path}')
    print(f'총 {len(df)}행 / 이미지 {len(fused_data)}장 (예측 0건 이미지: {n_empty}장)')
    if n_empty:
        print('⚠ 예측이 하나도 없는 이미지가 있습니다. score_thr를 낮추거나 해당 이미지를 육안 확인하세요.')
    return df

# 파일명 -> image_id 매핑을 눈으로 먼저 확인 (규칙이 다르면 extract_image_id 수정)
for d in fused_data[:5]:
    print(d['file_name'], '->', extract_image_id(d['file_name']))

submission_df = make_submission(fused_data, label2cat, SUBMIT_SCORE_THR, SUBMISSION_PATH)
submission_df.head(10)

## 산출물

- `/kaggle/working/outputs/large_task3_full74_masked_ep{누적:02d}_last.pth` — full-data 학습 RF-DETR Large 체크포인트
  (**마지막 epoch** 기준. 최종 제출 WBF 앙상블 노트북에서 task2-masked 5-fold 체크포인트 등과 함께 사용)
- `/kaggle/working/outputs/label_map.json` — 라벨(1~74) ↔ 원본 category_id 매핑 (앙상블 노트북 참조용)
- `/kaggle/working/outputs/..._history.png` — 청크별 학습곡선
- `/kaggle/working/submission_task3_large_task3_full74_masked.csv` — 단일모델 참고용 제출 CSV
  (Kaggle 스코어로 단독 성능을 가늠할 수 있음. 최종 제출은 별도 앙상블 노트북에서)
